In [1]:
import os
from dotenv import load_dotenv

load_dotenv(
    r"C:\Users\shekh\OneDrive\Desktop\Tutorial Agentic AI\.env",
    override=True
)

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found in .env"

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [2]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\shekh\AppData\Local\Temp\ipykernel_29568\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)

pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")

Loaded 9 pages from the PDF.


In [4]:
if pages:
    print("First page preview:\n")
    print(pages[0].page_content[:500])
else:
    print("No pages were extracted from the PDF.")

First page preview:

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

In [7]:
chunks = splitter.split_documents(pages)

print(f"Original pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Original pages: 9
Total chunks: 37


In [8]:
if chunks:
    print("Example chunk:\n")
    print(chunks[min(5, len(chunks) - 1)].page_content)

    average_length = sum(
        len(chunk.page_content) for chunk in chunks
    ) // len(chunks)

    print(f"\nAverage chunk length: {average_length} characters")
else:
    print("No chunks were created.")

Example chunk:

Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.

Average chunk length: 504 characters


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [10]:
print("Loading the embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading the embedding model...


C:\Users\shekh\OneDrive\Desktop\Tutorial Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2028.11it/s]


Embedding model loaded successfully!


In [11]:
print("Generating embeddings and storing chunks...")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="telecom_knowledge"
)

print("Vector store created successfully!")
print(f"Vectors stored: {vector_store._collection.count()}")

Generating embeddings and storing chunks...
Vector store created successfully!
Vectors stored: 37


In [12]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


In [13]:
test_query = "What is VoLTE and how does it improve call quality?"

retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks\n")

for i, doc in enumerate(retrieved, start=1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call ca

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

In [15]:
def format_docs(docs):
    return "\n\n---\n\n".join(
        doc.page_content for doc in docs
    )

In [16]:
SYSTEM_PROMPT = """
You are a helpful telecom assistant.

Answer the user's question using the supplied context.

Rules:
1. Base factual answers on the provided context.
2. If the context does not contain enough information,
   say that the available document does not provide the answer.
3. Do not invent telecom policies, charges, or procedures.
4. Explain technical concepts in simple language.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

print("Prompt template created successfully!")

Prompt template created successfully!


In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=0
)

print("Groq model initialized successfully!")

Groq model initialized successfully!


In [18]:
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled successfully!")

RAG chain assembled successfully!


In [19]:
question = (
    "How does international roaming work "
    "and what charges should I expect?"
)

print("Question:", question)

answer = chain.invoke(question)

print("\nAnswer:")
print(answer)

Question: How does international roaming work and what charges should I expect?

Answer:
**How international roaming works**

1. **You leave your home network’s coverage area** – your phone looks for a partner (visited) network in the country you’re in.  
2. **The visited network authenticates you** using an inter‑operator signalling protocol (SS7 or Diameter).  
3. **Your home network validates the subscription** and decides what services you’re allowed to use.  
4. **All voice, data and SMS traffic is tunneled back to your home network** so that it can be billed there.  
   - Because the traffic has to travel over that tunnel, you may notice a little extra latency compared with using the network locally.

**What charges you can expect**

Our company groups the world into three **roaming zones**, each with its own price level:

| Roaming Zone | Typical Countries | Relative Cost |
|--------------|-------------------|---------------|
| **Zone A**   | EU, UK, Australia, New Zealand | Low

In [20]:
print("Telecom RAG Assistant")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    if not question:
        continue

    try:
        answer = chain.invoke(question)

        print("\nAnswer:")
        print(answer)
        print("\n" + "-" * 60 + "\n")

    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")

Telecom RAG Assistant
Type 'quit' to exit.



Your question:  what is 4g mean



Answer:
**4G** (pronounced “four‑gee”) is the fourth generation of mobile‑phone network technology.  

- **What it does:** It provides much faster data speeds than earlier generations—typically **20 – 150 megabits per second** for downloads, with very low latency (under 50 ms).  
- **How it works:** 4G networks are **fully IP‑based**, meaning that everything—including voice calls—is carried as data packets. This is called **VoLTE (Voice over LTE)**, so you don’t need a separate circuit‑switched channel for voice.  
- **Why it matters:** The higher speed and lower delay let you stream high‑definition video, play online games, video‑chat, and use data‑intensive apps smoothly on your smartphone.  

In short, 4G is the modern mobile network that makes your phone feel more like a fast, always‑on broadband connection.

------------------------------------------------------------



Your question:  quit


Goodbye!
